# Live League Games

In [2]:
%load_ext autoreload
%autoreload 2
import requests
from dotenv import load_dotenv
import os
import pandas as pd
from datetime import datetime as dt
from sqlalchemy.engine import create_engine, URL
import numpy as np
import json
from retry import retry 
pd.set_option("display.max_columns", 100)
pd.set_option('display.max_rows', 50)
from src.config import ROOT_DIR

In [3]:
# Steam API constants
STEAM_URL = 'http://api.steampowered.com/'
LIVE_LEAGUE_GAMES = 'IDOTA2Match_570/GetLiveLeagueGames/v1'
REAL_TIME_STATS = 'IDOTA2MatchStats_570/GetRealtimeStats/v1' # Requires server_steam_id 

load_dotenv()
API_KEY = os.getenv('STEAM_API')

### Fetching premium and professional leagues' match details

In [4]:
# Function to retrieve Json from Steam WebAPI
session = requests.Session()
session.params.update({'key': API_KEY})

@retry(tries=3, delay=2)
def fetch_live_league_games():
    try:
        url = f'{STEAM_URL}{LIVE_LEAGUE_GAMES}'
        res = session.get(url)
        match_details = res.json()
        if not match_details:
            raise ValueError("Empty dictionary, retrying...")
        else:
            return match_details
    except Exception as err:
        print("Did not get a response, retrying...")
        raise

In [5]:
game_data = fetch_live_league_games()
games = game_data['result']['games']

In [6]:
games_df = pd.DataFrame(games)
games_df

,players,radiant_team,dire_team,lobby_id,match_id,spectators,league_id,league_node_id,stream_delay_s,radiant_series_wins,dire_series_wins,series_type,scoreboard
0,"[{'account_id': 938076668, 'name': 'Spell', 'h...","{'team_name': 'aut_chayhana', 'team_id': 97356...","{'team_name': 'tajikistan', 'team_id': 9651709...",29273295000037345,8261962712,2,18007,203,300,0,0,1,"{'duration': 1623, 'roshan_respawn_timer': 37,..."
1,"[{'account_id': 336764304, 'name': 'Kitsune', ...","{'team_name': 'Ozon Noobs', 'team_id': 9730204...","{'team_name': 'Team Mira', 'team_id': 9735701,...",29273294998102406,8261964804,1,18007,198,300,0,0,1,"{'duration': 1616.333251953125, 'roshan_respaw..."
2,"[{'account_id': 91447384, 'name': 'Dellexi', '...","{'team_name': 'Два ствола и пакет', 'team_id':...","{'team_name': 'RoadToTI', 'team_id': 9730142, ...",29273295012371022,8261976811,0,18007,200,300,0,0,1,"{'duration': 1009.5667114257812, 'roshan_respa..."
3,"[{'account_id': 460402443, 'name': 'Креведка',...","{'team_name': 'Химикал Брот', 'team_id': 97308...","{'team_name': 'Ghostbusters', 'team_id': 91741...",29273295016439334,8261971149,0,18007,202,120,0,0,1,"{'duration': 1622.2335205078125, 'roshan_respa..."
4,"[{'account_id': 166203463, 'name': 'pierettaap...","{'team_name': 'CHICAGO SKUFFS', 'team_id': 950...","{'team_name': 'OZON.KAZAN', 'team_id': 9330891...",29273295005044472,8261967020,0,18007,199,120,0,0,1,"{'duration': 1674.3668212890625, 'roshan_respa..."
5,"[{'account_id': 1452978908, 'name': '小可爱吴卓佩', ...","{'team_name': '浙江大学JSY', 'team_id': 9749591, '...","{'team_name': '北京航空航天大学六代战机', 'team_id': 97302...",29273295024761520,8261986955,27,18047,0,300,0,0,0,"{'duration': 392.2332763671875, 'roshan_respaw..."
6,"[{'account_id': 140738059, 'name': '你发问前似乎从不思考...",NaN,"{'team_name': 'COG', 'team_id': 9743555, 'team...",29273295036328448,8261987228,22,18047,0,300,0,0,0,"{'duration': 470.0999755859375, 'roshan_respaw..."
7,"[{'account_id': 129406533, 'name': '第19使徒: :II...","{'team_name': '哈尔滨工业大学', 'team_id': 6121402, '...","{'team_name': 'ZSTU', 'team_id': 9741248, 'tea...",29273295037828546,8261986208,7,18047,0,300,0,0,0,"{'duration': 428.76666259765625, 'roshan_respa..."
8,"[{'account_id': 111957685, 'name': '我先鼠了', 'he...","{'team_name': '青鸟战队', 'team_id': 9742000, 'tea...","{'team_name': 'USST-BLACK CHICKEN', 'team_id':...",29273295026372999,8261983238,7,18047,0,300,0,0,0,"{'duration': 591.8333740234375, 'roshan_respaw..."
9,"[{'account_id': 1097654166, 'name': '水平和脾气一样差'...","{'team_name': '北京航空航天大学北京一号战队', 'team_id': 974...",NaN,29273295036988136,8261999462,2,18047,0,300,0,0,0,"{'duration': 0, 'roshan_respawn_timer': 0, 'ra..."


In [7]:
match = games_df.head(1)

In [8]:
json.loads(match.to_json(orient='records', date_format='iso'))

[{'players': [{'account_id': 938076668,
    'name': 'Spell',
    'hero_id': 0,
    'team': 4},
   {'account_id': 860271515, 'name': 'null', 'hero_id': 53, 'team': 1},
   {'account_id': 911479785, 'name': 'ttQ', 'hero_id': 100, 'team': 1},
   {'account_id': 78392868, 'name': 'Real_Good', 'hero_id': 39, 'team': 1},
   {'account_id': 149014813, 'name': 'Artful-', 'hero_id': 135, 'team': 1},
   {'account_id': 358426134, 'name': 'HAOS', 'hero_id': 138, 'team': 1},
   {'account_id': 1073759713, 'name': 'чижик', 'hero_id': 22, 'team': 0},
   {'account_id': 130169737, 'name': 'uselesscloud', 'hero_id': 64, 'team': 0},
   {'account_id': 141557630, 'name': 'hotaken', 'hero_id': 6, 'team': 0},
   {'account_id': 129231690, 'name': 'Paranoia', 'hero_id': 29, 'team': 0},
   {'account_id': 129869699, 'name': 'St_Ilia', 'hero_id': 87, 'team': 0}],
  'radiant_team': {'team_name': 'aut_chayhana',
   'team_id': 9735655,
   'team_logo': 38950614858552719,
   'complete': True},
  'dire_team': {'team_name':

In [9]:
# Import the list of premium and professional league games id

import yaml

file_path = os.path.join(ROOT_DIR, f'constants/league_ids.yml')

with open(file_path, 'r') as file:
    content = yaml.safe_load(file) or {}
    if 'PREMIUM_LEAGUES' in content:
        premium_leagues = content['PREMIUM_LEAGUES']
    if 'PROFESSIONAL_LEAGUES' in content:
        professional_leagues = content['PROFESSIONAL_LEAGUES']
        
premium_list = list(premium_leagues.values())
professional_list = list(professional_leagues.values())



In [7]:
from src.pydantic_models.match import Match
from src.pydantic_models.live_league_games import LiveLeagueGames

In [8]:
live_league_games = []

for row in games:
    
    league_id = row.get('league_id', np.nan)
    if league_id in premium_list + professional_list:
    
        game_data = LiveLeagueGames(**row)
        
        # Populate common fields
        match_data = {
            'match_id': game_data.match_id,
            'radiant_team_id': game_data.radiant_team.team_id,
            'radiant_name': game_data.radiant_team.team_name,
            'dire_team_id': game_data.dire_team.team_id,
            'dire_name': game_data.dire_team.team_name,
            'duration': game_data.scoreboard.duration,
            'start_time': int(dt.now().timestamp())
        }
        
        # Populate player data
        for team in ['radiant', 'dire']:
            faction = getattr(game_data.scoreboard, team)
            for player in faction.players:
                slot = player.player_slot
                player_data = {
                    f"slot_{slot}_account_id": player.account_id,
                    f"slot_{slot}_hero_id": player.hero_id
                } 
                match_data.update(player_data)
                
        live_league_games.append(Match(**match_data))
                

    
if len(live_league_games) == 0:
    print("No premium or professional games right now")
else:
    print(len(live_league_games))
    print(live_league_games)   


No premium or professional games right now


In [20]:
live_league_games

[Match(match_id=8259083479, radiant_name='Romashku', radiant_team_id=9735994, dire_name='Nethercore', dire_team_id=9593609, start_time=1744966361, duration=585.5668334960938, radiant_win=None, slot_0_hero_id=5, slot_1_hero_id=79, slot_2_hero_id=16, slot_3_hero_id=49, slot_4_hero_id=106, slot_128_hero_id=46, slot_129_hero_id=108, slot_130_hero_id=100, slot_131_hero_id=26, slot_132_hero_id=126, slot_0_account_id=337792133, slot_1_account_id=1674492970, slot_2_account_id=1885001216, slot_3_account_id=1885118437, slot_4_account_id=1683752803, slot_128_account_id=1029972951, slot_129_account_id=314272971, slot_130_account_id=1712916118, slot_131_account_id=1695617010, slot_132_account_id=1810099486)]